In [1]:
# import needed libraries
import os
import numpy as np
import open3d as o3d

In [2]:
# Configuration
USED_PCD_ID = 300

In [9]:
# get current working directory and find pcd files
print('Working directory:', os.getcwd())
path_points = os.getcwd() + '/points'

# check if path exists
if not os.path.exists(path_points):
    print('Path does not exist, please make sure your path is correct')
    exit()

# find all pcd files in the path
pcd_file_cnt = 0
files = os.listdir(path_points)
pcd_files = []

# count the number of pcd files
for file in files:
    if file.endswith('.pcd'):
        pcd_file_cnt += 1
        pcd_files.append(file)

# counter all files size totaly
total_size = 0
for file in pcd_files:
    total_size += os.path.getsize(os.getcwd() + '/points/' + file)

pcd_file_name = pcd_files[USED_PCD_ID]
#print pcd files info
print(f'Total {pcd_file_cnt} pcd files found, total size: {total_size / 1024 / 1024:.2f} MB')
print(f'Using No.{USED_PCD_ID} pcd file: {pcd_file_name}')

Working directory: /root/AC1/autopath
Total 741 pcd files found, total size: 475.11 MB
Using No.300 pcd file: 1771498195.700643062.pcd


In [10]:
# cpoy target pcd file to used_pcd folder
if not os.path.exists(os.getcwd() + '/used_pcd'):
    print('Create used_pcd folder')
    os.makedirs(os.getcwd() + '/used_pcd')

# Check if the file already exists in the used_pcd folder
if os.path.exists(os.getcwd() + '/used_pcd/' + pcd_file_name):
    print(f'File {pcd_file_name} already exists in used_pcd folder, Checking if it is the same file...')
    if os.path.getsize(os.getcwd() + '/used_pcd/' + pcd_file_name) == os.path.getsize(path_points + '/' + pcd_file_name):
        print(f'File {pcd_file_name} is the same file, no need to copy')
    else:
        print(f'File {pcd_file_name} is different file, copying...')
        os.remove(os.getcwd() + '/used_pcd/' + pcd_file_name)
        os.rename(path_points + '/' + pcd_file_name, os.getcwd() + '/used_pcd/' + pcd_file_name)
else:
    print(f'File {pcd_file_name} does not exist in used_pcd folder, copying...')
    # open source file in binary mode
    with open(path_points + '/' + pcd_file_name, 'rb') as source:
        # open destination file in binary mode
        with open(os.getcwd() + '/used_pcd/' + pcd_file_name, 'wb') as destination:
            # copy the contents of the source file to the destination file
            destination.write(source.read())


File 1771498195.700643062.pcd already exists in used_pcd folder, Checking if it is the same file...
File 1771498195.700643062.pcd is the same file, no need to copy


In [10]:
# Show point cloud
def show_point_cloud(pcdx, color=[1, 0, 0]):
    # 等价于手动给所有点的colors属性赋值相同的RGB
    num_points = len(pcdx.points)
    # 创建和点数量一致的颜色数组，所有点都是[1,0,0]（红色）
    color_array = np.ones((num_points, 3)) * color
    # 赋值给colors属性，覆盖Open3D的默认着色逻辑
    pcdx.colors = o3d.utility.Vector3dVector(color_array)

    # Visualize points cloud
    o3d.visualization.draw_geometries([pcdx])

In [11]:
# Read point cloud
pcd = o3d.io.read_point_cloud(pcd_files[USED_PCD_ID])

# remove nan points and infinite points
pcd = pcd.remove_non_finite_points(remove_nan=True, remove_infinite=True)

# Visualize if needed
# o3d.visualization.draw_geometries([pcd])
show_point_cloud(pcd)

# print pcd info
print("Valid points:", len(pcd.points))

Valid points: 17628


In [6]:
# Downsample
downpcd = pcd.voxel_down_sample(voxel_size=0.1)
# o3d.visualization.draw_geometries([downpcd])
# print downpcd info
print(f"Downsampled points: {len(downpcd.points)}, downsample ratio: {len(downpcd.points) / len(pcd.points) * 100:.2f}%", )

Downsampled points: 8180, downsample ratio: 46.40%


In [9]:
# Remove outliers
cl, ind = downpcd.remove_statistical_outlier(nb_neighbors=10, std_ratio=1.0)
print(cl)
# o3d.visualization.draw_geometries([cl])

# print(ind)
filtered_pcd = downpcd.select_by_index(ind)
print(filtered_pcd)

# Visualize if needed
o3d.visualization.draw_geometries([filtered_pcd])

print(f"原始点云数量: {len(pcd.points)}, 预处理后: {len(filtered_pcd.points)}")

PointCloud with 7185 points.
PointCloud with 7185 points.
原始点云数量: 17628, 预处理后: 7185


In [8]:
# 使用RANSAC拟合平面
plane_model, inliers = pcd.segment_plane(
    distance_threshold=0.05,
    ransac_n=3,
    num_iterations=1000
)
[a, b, c, d] = plane_model
print(f"拟合平面方程: {a:.4f}x + {b:.4f}y + {c:.4f}z + {d:.4f} = 0")

# 筛选水平面（法向量z分量绝对值接近1，即法向量接近Z轴）
normal = np.array([a, b, c])
normal = normal / np.linalg.norm(normal)  # 归一化法向量
z_component = abs(normal[2])
if z_component < 0.9:  # 法向量z分量小于0.9，认为不是水平面
    raise ValueError(f"检测到的平面不是水平面（法向量z分量: {z_component:.4f}）")

# 提取平面内的点云
plane_pcd = pcd.select_by_index(inliers)
# plane_pcd.paint_uniform_color([1, 0, 0])  # 红色标记水平面
o3d.visualization.draw_geometries([plane_pcd])

# 计算平面中心坐标
plane_points = np.asarray(plane_pcd.points)
plane_center = np.mean(plane_points, axis=0)
print(f"水平面中心坐标: {plane_center}")

拟合平面方程: -0.1446x + 0.1025y + 0.9842z + 1.8426 = 0
水平面中心坐标: [ 4.14501646 -1.01856568 -1.15692607]


In [ ]:
import matplotlib.pyplot as plt

# -------------------------- 2. 检测水平面 --------------------------
def detect_horizontal_plane(pcd, distance_threshold=0.05, ransac_n=3, num_iterations=1000):
    """
    从点云中检测水平面（法向量接近Z轴）
    :param pcd: 预处理后的点云对象
    :param distance_threshold: 点到平面的距离阈值
    :param ransac_n: RANSAC每次采样的点数
    :param num_iterations: RANSAC迭代次数
    :return: 平面参数（ax+by+cz+d=0）、平面点云、平面中心坐标
    """
    # 使用RANSAC拟合平面
    plane_model, inliers = pcd.segment_plane(
        distance_threshold=distance_threshold,
        ransac_n=ransac_n,
        num_iterations=num_iterations
    )
    [a, b, c, d] = plane_model
    print(f"拟合平面方程: {a:.4f}x + {b:.4f}y + {c:.4f}z + {d:.4f} = 0")
    
    # 筛选水平面（法向量z分量绝对值接近1，即法向量接近Z轴）
    normal = np.array([a, b, c])
    normal = normal / np.linalg.norm(normal)  # 归一化法向量
    z_component = abs(normal[2])
    if z_component < 0.9:  # 法向量z分量小于0.9，认为不是水平面
        raise ValueError(f"检测到的平面不是水平面（法向量z分量: {z_component:.4f}）")
    
    # 提取平面内的点云
    plane_pcd = pcd.select_by_index(inliers)
    # plane_pcd.paint_uniform_color([1, 0, 0])  # 红色标记水平面
    
    # 计算平面中心坐标
    plane_points = np.asarray(plane_pcd.points)
    plane_center = np.mean(plane_points, axis=0)
    print(f"水平面中心坐标: {plane_center}")
    
    return plane_model, plane_pcd, plane_center

# -------------------------- 3. 规划降落路径 --------------------------
def plan_landing_path(plane_center, safe_height=2.0, step_num=50):
    """
    规划无人机降落路径：原点 → 目标点正上方（安全高度） → 垂直降落至平面中心
    :param plane_center: 水平面中心坐标 [x, y, z]
    :param safe_height: 目标点正上方的安全高度（相对于平面）
    :param step_num: 路径点数量（均分）
    :return: 降落路径点云、路径点坐标数组
    """
    # 路径关键点
    start_point = np.array([0, 0, 0])  # 起点（原点）
    hover_point = np.array([plane_center[0], plane_center[1], plane_center[2] + safe_height])  # 悬停点（安全高度）
    land_point = plane_center  # 降落点（平面中心）
    
    # 生成分段平滑路径
    # 第一段：原点到悬停点（直线）
    path_1 = np.linspace(start_point, hover_point, step_num // 2)
    # 第二段：悬停点到降落点（垂直下降）
    path_2 = np.linspace(hover_point, land_point, step_num // 2)
    # 合并路径
    path_points = np.vstack((path_1, path_2))
    
    # 创建路径点云（蓝色标记）
    path_pcd = o3d.geometry.PointCloud()
    path_pcd.points = o3d.utility.Vector3dVector(path_points)
    path_pcd.paint_uniform_color([0, 0, 1])  # 蓝色标记路径
    
    print(f"降落路径规划完成：")
    print(f"  - 起点: {start_point}")
    print(f"  - 悬停点: {hover_point} (安全高度: {safe_height}m)")
    print(f"  - 降落点: {land_point}")
    
    return path_pcd, path_points

# -------------------------- 4. 可视化结果 --------------------------
def visualize_result(pcd, plane_pcd, path_pcd):
    """
    可视化原始点云、水平面、降落路径
    """
    # 原始点云设为灰色
    pcd.paint_uniform_color([0.5, 0.5, 0.5])
    # 创建坐标系（原点）
    coordinate_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0, origin=[0, 0, 0])
    # 可视化
    o3d.visualization.draw_geometries([pcd, plane_pcd, path_pcd, coordinate_frame],
                                      window_name="点云水平面检测与降落路径规划",
                                      width=1280, height=720)


In [8]:
# 检测水平面
plane_model, plane_pcd, plane_center = detect_horizontal_plane(filtered_pcd)

# 规划降落路径
path_pcd, path_points = plan_landing_path(plane_center, safe_height=2.0)

# 可视化结果
visualize_result(pcd, plane_pcd, path_pcd)

拟合平面方程: -0.1446x + 0.1025y + 0.9842z + 1.8427 = 0


: 